In [1]:
!pip install -q "lm_eval[ifeval]"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import gc, torch
import lm_eval

MODELS = {
    "Base": "allenai/OLMo-2-0425-1B",
    "SFT": "allenai/OLMo-2-0425-1B-SFT",
    "DPO": "allenai/OLMo-2-0425-1B-DPO",
    "Instruct": "allenai/OLMo-2-0425-1B-Instruct",
}
TASKS = ["gsm8k", "ifeval", "truthfulqa_mc2"]
LIMIT = 200

all_res = {}
for name, path in MODELS.items():
    all_res[name] = lm_eval.simple_evaluate(
        model="hf",
        model_args=f"pretrained={path},dtype=float16",
        tasks=TASKS,
        num_fewshot=0,
        limit=LIMIT,
        batch_size=8,
        apply_chat_template=(name != "Base"),  # base model doesnt have template
        log_samples=True,
    )
    gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (47 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Running loglikelihood requests: 100%|██████████| 1476/1476 [00:51<00:00, 28.61it/s]


In [3]:
import pandas as pd

cols = {
    "GSM8K (flexible)": ("gsm8k", "exact_match,flexible-extract"),
    "GSM8K (strict)": ("gsm8k", "exact_match,strict-match"),
    "IFEval prompt-strict": ("ifeval", "prompt_level_strict_acc,none"),
    "IFEval inst-strict": ("ifeval", "inst_level_strict_acc,none"),
    "TruthfulQA MC2": ("truthfulqa_mc2", "acc,none"),
}
table = pd.DataFrame({m: {k: r["results"][t][key] for k, (t, key) in cols.items()}
                      for m, r in all_res.items()}).T.round(3)
table

,GSM8K (flexible),GSM8K (strict),IFEval prompt-strict,IFEval inst-strict,TruthfulQA MC2
Base,0.040,0.0,0.095,0.195,0.388
SFT,0.475,0.0,0.395,0.535,0.448
DPO,0.465,0.0,0.535,0.673,0.466
Instruct,0.670,0.0,0.585,0.701,0.486


In [4]:
# 2 prompts of GSM8K and IFEval at each checkpoint
for task in ["gsm8k", "ifeval"]:
    for name, r in all_res.items():
        for s in r["samples"][task][:2]:
            prompt = s["doc"].get("question", s["doc"].get("prompt"))
            print(f"=== {task} | {name} ===\nPROMPT: {prompt}\nOUTPUT: {s['resps'][0][0][:500]}\n")

=== gsm8k | Base ===
PROMPT: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
OUTPUT:  16 eggs x $2 = $32


=== gsm8k | Base ===
PROMPT: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
OUTPUT:  4

=== gsm8k | SFT ===
PROMPT: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
OUTPUT: She eats 3 eggs for breakfast and bakes 4 muffins for her friends so she uses 3 + 4 = 7 eggs per day. She has 16 - 7 = 9 eggs left to sell at the farmers' market. She sells each egg for $2 so she 